In [13]:
# comparing the counts matirx to see the reads mapped to actual genes ChrM and ChrC
# (a) GENOME ALIGNMENT  - where samtools idxstats says each read landed. 
# (b) GENE ASSIGNMENT   - UMIs per gene from the alevin-fry matrix

# map gene tp chr
prefix_to_chrom = {
    "AT1G": "Chr1",
    "AT2G": "Chr2",
    "AT3G": "Chr3",
    "AT4G": "Chr4",
    "AT5G": "Chr5",
    "ATCG": "ChrC",
    "ATMG": "ChrM",
}

AF = "/data1/bfernando/DE_project/alvinfry"
SAMPLE = "Col-0_quant"
cols_file = f"{AF}/{SAMPLE}/alevin/quants_mat_cols.txt"
print(cols_file)
mtx_file  = f"{AF}/{SAMPLE}/alevin/quants_mat.mtx"

column_chrom = {}
line_number = 0

# split genes by chr they belong to
for line in open(cols_file):
    line_number = line_number + 1
    name = line.strip()
    gene_id = name.split(".")[0]        # "ATCG00070.Araport11.447-U" -> "ATC

    if gene_id == "AT4G28110" and "Fusion" in name:
        column_chrom[line_number] = "pFusionMYB41"
    else:
        column_chrom[line_number] = prefix_to_chrom.get(gene_id[:4], "other")


# --- Compute the sum of UMIs per chromosome 
umi_per_chrom = {}
total_umi = 0.0

with open(mtx_file) as fh:
    fh.readline()
    fh.readline()
    fh.readline()
    for line in fh:
        parts = line.split()
        column = int(parts[1])
        umis = float(parts[2])

        chrom = column_chrom[column]
        umi_per_chrom[chrom] = umi_per_chrom.get(chrom, 0) + umis
        total_umi = total_umi + umis

# --- read counts, from:  samtools idxstats Col-0_v4/outs/possorted_genome_bam.bam
reads_per_chrom = {
    "Chr1":          18338962,
    "Chr2":         146114338,
    "Chr3":         217895296,
    "Chr4":          10835887,
    "Chr5":          16438079,
    "ChrC":           3496063,
    "ChrM":          36395838,
    "pFusionMYB41":        67,
}

total_reads = 0
for chrom in reads_per_chrom:
    total_reads = total_reads + reads_per_chrom[chrom]

# --- print side by side ----------------------------------------------------
print("Col-0 - GENOME ALIGNMENT (reads) vs GENE ASSIGNMENT (UMIs)\n")
print(f"{'':14} {'reads':>13} {'% reads':>9}   {'gene UMIs':>12} {'% UMIs':>8}")

order = ["Chr1", "Chr2", "Chr3", "Chr4", "Chr5", "ChrC", "ChrM", "pFusionMYB41"]
for chrom in order:
    reads = reads_per_chrom[chrom]
    umis = umi_per_chrom.get(chrom, 0)

    pct_reads = reads / total_reads * 100
    pct_umis = umis / total_umi * 100

    print(f"{chrom:14} {reads:>13,} {pct_reads:>8.1f}%   "
          f"{int(umis):>12,} {pct_umis:>7.2f}%")

print(f"{'TOTAL':14} {total_reads:>13,} {'':>9}   {int(total_umi):>12,}")







/data1/bfernando/DE_project/alvinfry/Col-0_quant/alevin/quants_mat_cols.txt
Col-0 - GENOME ALIGNMENT (reads) vs GENE ASSIGNMENT (UMIs)

                       reads   % reads      gene UMIs   % UMIs
Chr1              18,338,962      4.1%      4,050,842   23.03%
Chr2             146,114,338     32.5%      2,882,036   16.38%
Chr3             217,895,296     48.5%      3,470,278   19.73%
Chr4              10,835,887      2.4%      2,385,493   13.56%
Chr5              16,438,079      3.7%      3,961,160   22.52%
ChrC               3,496,063      0.8%        291,636    1.66%
ChrM              36,395,838      8.1%        550,989    3.13%
pFusionMYB41              67      0.0%              1    0.00%
TOTAL            449,514,530               17,592,435


In [9]:
# Check how many UMIs per barcode are unspliced.
# Ambient RNA (cytoplasm )and MatureRNA differ in splicing.
# within nuclei the mRNA would be in its pre-mRNA form with introns.

# So if this is snRNA-seq, %U should rise as barcodes get bigger.

AF = "/data1/bfernando/DE_project/alvinfry"
SAMPLE = "Col-0_quant"
cols_file = f"{AF}/{SAMPLE}/alevin/quants_mat_cols.txt"
print(cols_file)
mtx_file  = f"{AF}/{SAMPLE}/alevin/quants_mat.mtx"

n_columns = 0
for line in open(cols_file):
    # print(line)
    n_columns = n_columns + 1

print("Total genes (USA mode):",n_columns)
n_genes = n_columns // 3
print("Genes : ",n_genes)

# The File : /data1/bfernando/DE_project/alvinfry/Col-0_quant/alevin/quants_mat_cols.txt
# its stacked with order 1: Splicied gene Names 2: Unsplicied 3: Ambiguous
end_of_S = n_genes
end_of_U = n_genes * 2 
print(f"{n_genes:,} genes -> S ends at {end_of_S:,}, U ends at {end_of_U:,}")

###### Count the barcodes for each reference section

# per-barcode totals
S_per_bc = {}
U_per_bc = {}
A_per_bc = {}
T_per_bc = {} # total UMI across S| U and A

print(mtx_file)
# in mtx_file --> rows = barcodes and cols = genes 
# dimensions - 192259 82968 14268087, from above we have total genes as 82968

with open(mtx_file) as fh:
    fh.readline()                 # header line 1
    fh.readline()                 # header line 2
    fh.readline()                 # dimension line
    for line in fh:
        parts = line.split()
        barcode = int(parts[0])
        column  = int(parts[1])
        umis    = float(parts[2])

        if column <= end_of_S:
            S_per_bc[barcode] = S_per_bc.get(barcode, 0) + umis
        elif column <= end_of_U:
            U_per_bc[barcode] = U_per_bc.get(barcode, 0) + umis
        else:
            A_per_bc[barcode] = A_per_bc.get(barcode, 0) + umis

        T_per_bc[barcode] = T_per_bc.get(barcode, 0) + umis



/data1/bfernando/DE_project/alvinfry/Col-0_quant/alevin/quants_mat_cols.txt
Total genes (USA mode): 82968
Genes :  27656
27,656 genes -> S ends at 27,656, U ends at 55,312
/data1/bfernando/DE_project/alvinfry/Col-0_quant/alevin/quants_mat.mtx


In [10]:
# group the barcodes into size bins
bins = [(0, 50), (50, 100), (100, 500), (500, 1000), (1000, 5000), (5000, None)]

print(f"\n{SAMPLE} - unspliced content vs barcode size")
print(f"{'UMI bin':>16} {'barcodes':>10} {'%U':>7} {'%U+A':>7}")
for low, high in bins:
    n_barcodes = 0
    bin_total  = 0.0
    bin_U      = 0.0
    bin_A      = 0.0

    for barcode in T_per_bc:
        total = T_per_bc[barcode]

        in_bin = total > low
        if high is not None and total > high:
            in_bin = False

        if in_bin:
            n_barcodes = n_barcodes + 1
            bin_total  = bin_total + total
            bin_U      = bin_U + U_per_bc.get(barcode, 0)
            bin_A      = bin_A + A_per_bc.get(barcode, 0)

    if n_barcodes == 0:
        continue

    pct_U  = bin_U / bin_total * 100
    pct_UA = (bin_U + bin_A) / bin_total * 100
    label  = f"{low}-{high if high is not None else 'inf'}"
    print(f"{label:>16} {n_barcodes:>10,} {pct_U:>6.1f}% {pct_UA:>6.1f}%")



Col-0_quant - unspliced content vs barcode size
         UMI bin   barcodes      %U    %U+A
            0-50     55,537    5.5%   13.7%
          50-100    107,880    6.1%   14.8%
         100-500     25,634    5.7%   13.7%
        500-1000      1,772    5.3%   12.6%
       1000-5000      1,013    5.3%   12.3%
        5000-inf        149    3.6%    9.5%


Barcodes with 0-100 UMI are mostly empty droplets holding ambient RNA;
barcodes above ~1000 UMI are the cell-like ones .

nuclear RNA should be enriched for unspliced transcripts compared with cytoplasm-derived ambient RNA.
If these were nuclei, %U should rise with barcode size.

It doesn't - %U is ~5-6% across every bin, and the 149 largest barcodes are
the lowest at 3.6% U (with A at 5.9%, so 90.5% spliced). The cell-like
barcodes show no more unspliced signal than the background.
